# Лабораторная работа №4. Использование библиотеки pandas
## Задание 2. Отток клиентов оператора связи
Файл `telecom_churn.csv` должен находиться рядом с блокнотом. Необходим Python с pandas.
В обозначениях ниже `False` — активный клиент, `True` — потерянный клиент.

## 1. Загрузка, общая информация и пропуски

In [1]:
import pandas as pd
from IPython.display import display

df = pd.read_csv('telecom_churn.csv')
# Приводим Yes/No к единому регистру.
df['International plan'] = df['International plan'].str.strip().str.lower()
df.info()
display(df.describe().round(2))
missing = df.isna().sum()
display(missing.to_frame('Пропуски'))
print('Отсутствующие данные есть.' if missing.sum() else 'Отсутствующих данных нет.')

<class 'pandas.DataFrame'>
RangeIndex: 3333 entries, 0 to 3332
Data columns (total 20 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   State                   3333 non-null   str    
 1   Account length          3333 non-null   int64  
 2   Area code               3333 non-null   int64  
 3   International plan      3333 non-null   str    
 4   Voice mail plan         3333 non-null   str    
 5   Number vmail messages   3333 non-null   int64  
 6   Total day minutes       3333 non-null   float64
 7   Total day calls         3333 non-null   int64  
 8   Total day charge        3333 non-null   float64
 9   Total eve minutes       3333 non-null   float64
 10  Total eve calls         3333 non-null   int64  
 11  Total eve charge        3333 non-null   float64
 12  Total night minutes     3333 non-null   float64
 13  Total night calls       3333 non-null   int64  
 14  Total night charge      3333 non-null   float64
 15

,Account length,Area code,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls
count,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00,3333.00
mean,101.06,437.18,8.10,179.78,100.44,30.56,200.98,100.11,17.08,200.87,100.11,9.04,10.24,4.48,2.76,1.56
std,39.82,42.37,13.69,54.47,20.07,9.26,50.71,19.92,4.31,50.57,19.57,2.28,2.79,2.46,0.75,1.32
min,1.00,408.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,23.20,33.00,1.04,0.00,0.00,0.00,0.00
25%,74.00,408.00,0.00,143.70,87.00,24.43,166.60,87.00,14.16,167.00,87.00,7.52,8.50,3.00,2.30,1.00
50%,101.00,415.00,0.00,179.40,101.00,30.50,201.40,100.00,17.12,201.20,100.00,9.05,10.30,4.00,2.78,1.00
75%,127.00,510.00,20.00,216.40,114.00,36.79,235.30,114.00,20.00,235.30,113.00,10.59,12.10,6.00,3.27,2.00
max,243.00,510.00,51.00,350.80,165.00,59.64,363.70,170.00,30.91,395.00,175.00,17.77,20.00,20.00,5.40,9.00


,Пропуски
State,0
Account length,0
Area code,0
International plan,0
Voice mail plan,0
Number vmail messages,0
Total day minutes,0
Total day calls,0
Total day charge,0
Total eve minutes,0


Отсутствующих данных нет.


## 2. Количество и доля активных и потерянных клиентов

In [2]:
counts = df['Churn'].value_counts().sort_index()
percentages = df['Churn'].value_counts(normalize=True).sort_index().mul(100)
display(pd.DataFrame({'Клиентов': counts, 'Доля, %': percentages}).round(2))

,Клиентов,"Доля, %"
Churn,,
False,2850,85.51
True,483,14.49


## 3. Средняя продолжительность звонка и первые 10 записей после сортировки
В основной расчёт включаем дневные, вечерние и ночные звонки — категории, перечисленные в задании.
В CSV также есть международные звонки; дополнительно рассчитываем вариант с ними, чтобы явно учесть трактовку «всех звонков».
Нулевое количество звонков заменяем на NA перед делением.

In [3]:
minutes = ['Total day minutes', 'Total eve minutes', 'Total night minutes']
calls = ['Total day calls', 'Total eve calls', 'Total night calls']
total_calls = df[calls].sum(axis=1)
df['Средняя длительность звонка'] = df[minutes].sum(axis=1).div(total_calls.replace(0, float('nan')))
df['Средняя длительность с международными'] = df[minutes + ['Total intl minutes']].sum(axis=1).div(
    df[calls + ['Total intl calls']].sum(axis=1).replace(0, float('nan')))
df = df.sort_values('Средняя длительность звонка', ascending=False)
with pd.option_context('display.max_columns', None):
    display(df.head(10))

,State,Account length,Area code,International plan,Voice mail plan,Number vmail messages,Total day minutes,Total day calls,Total day charge,Total eve minutes,Total eve calls,Total eve charge,Total night minutes,Total night calls,Total night charge,Total intl minutes,Total intl calls,Total intl charge,Customer service calls,Churn,Средняя длительность звонка,Средняя длительность с международными
985,NY,64,415,yes,No,0,346.8,55,58.96,249.5,79,21.21,275.4,102,12.39,13.3,9,3.59,1,True,3.693644,3.612245
2824,OR,81,415,no,No,0,324.7,48,55.20,236.4,82,20.09,187.6,78,8.44,13.1,5,3.54,0,True,3.599519,3.576526
244,VA,108,408,no,No,0,198.5,99,33.75,267.8,60,22.76,354.9,75,15.97,9.4,3,2.54,0,True,3.509402,3.504641
2321,AR,153,408,no,No,0,154.6,56,26.28,263.0,84,22.36,367.7,89,16.55,15.5,2,4.19,1,False,3.429258,3.466667
2033,NJ,162,408,no,No,0,217.6,87,36.99,279.0,71,23.72,250.7,65,11.28,10.4,4,2.81,2,True,3.351121,3.337885
1709,NV,96,408,no,No,0,197.7,68,33.61,250.5,53,21.29,181.2,67,8.15,10.5,3,2.84,3,False,3.347872,3.350262
2536,CT,119,408,no,No,0,294.2,100,50.01,232.5,53,19.76,195.0,64,8.78,9.0,1,2.43,0,True,3.325806,3.351835
1686,CT,163,408,no,Yes,40,231.9,56,39.42,211.8,91,18.00,268.5,74,12.08,12.3,3,3.32,2,False,3.222624,3.234375
649,MO,79,408,no,No,0,261.7,97,44.49,210.6,48,17.90,256.7,83,11.55,6.0,3,1.62,3,True,3.197368,3.181818
2289,MS,98,415,yes,Yes,23,245.5,54,41.74,292.7,83,24.88,184.0,90,8.28,10.8,7,2.92,1,False,3.181498,3.132479


## 4. Средняя продолжительность звонка по признаку оттока

In [4]:
duration = df.groupby('Churn')[['Средняя длительность звонка', 'Средняя длительность с международными']].mean()
display(duration.round(4))
active = duration.loc[False, 'Средняя длительность звонка']
lost = duration.loc[True, 'Средняя длительность звонка']
print(f'У потерянных клиентов средняя длительность выше на {lost-active:.3f} мин., или {(lost/active-1)*100:.2f}%.')

,Средняя длительность звонка,Средняя длительность с международными
Churn,,
False,1.9381,1.9424
True,2.0912,2.0974


У потерянных клиентов средняя длительность выше на 0.153 мин., или 7.90%.


Разница в длительности умеренная; одна средняя величина не позволяет надёжно разделять клиентов. Здесь сравниваются описательные показатели, статистическая значимость не проверялась.

## 5. Среднее число обращений в поддержку по признаку оттока

In [5]:
support = df.groupby('Churn')['Customer service calls'].mean()
display(support.to_frame('Среднее число обращений').round(3))
print(f'Потерянные клиенты обращались в {support.loc[True]/support.loc[False]:.2f} раза чаще, разница {support.loc[True]-support.loc[False]:.2f} обращения.')

,Среднее число обращений
Churn,
False,1.45
True,2.23


Потерянные клиенты обращались в 1.54 раза чаще, разница 0.78 обращения.


Различие в числе обращений заметнее, чем в длительности звонка. Это связь признаков в выборке; она сама по себе не доказывает, что обращения вызывают уход клиента.

## 6. Таблица сопряжённости: обращения в поддержку и отток

In [6]:
service_table = pd.crosstab(df['Customer service calls'], df['Churn']).reindex(columns=[False, True], fill_value=0)
service_table['Отток, %'] = service_table[True].div((service_table[False] + service_table[True])).mul(100)
display(service_table.round(2))
baseline = df['Churn'].mean() * 100
high_calls = service_table.index[service_table['Отток, %'].gt(40)]
print(f'Отток в целом: {baseline:.2f}%.')
print('Отток более 40% при числе обращений:', high_calls.tolist())

Churn,False,True,"Отток, %"
Customer service calls,,,
0,605,92,13.20
1,1059,122,10.33
2,672,87,11.46
3,385,44,10.26
4,90,76,45.78
5,26,40,60.61
6,8,14,63.64
7,4,5,55.56
8,1,1,50.00


Отток в целом: 14.49%.
Отток более 40% при числе обращений: [4, 5, 6, 7, 8, 9]


Малочисленные группы с большим числом обращений дают менее устойчивые оценки. Порог 40% взят из условия задания.

## 7. Таблица сопряжённости: международный роуминг и отток

In [7]:
plan_table = pd.crosstab(df['International plan'], df['Churn']).reindex(columns=[False, True], fill_value=0)
plan_table['Отток, %'] = plan_table[True].div((plan_table[False] + plan_table[True])).mul(100)
display(plan_table.round(2))
yes, no = plan_table['Отток, %'].loc[['yes', 'no']]
print(f'С роумингом отток {yes:.2f}%, без него {no:.2f}%. Разница: {yes-no:.2f} процентного пункта; отношение: {yes/no:.2f}.')

Churn,False,True,"Отток, %"
International plan,,,
no,2664,346,11.50
yes,186,137,42.41


С роумингом отток 42.41%, без него 11.50%. Разница: 30.92 процентного пункта; отношение: 3.69.


В этой выборке доля ушедших клиентов с международным роумингом существенно выше. Причинный вывод по этой таблице сделать нельзя.

## 8. Простой прогноз и ошибки первого и второго рода
Правило: прогнозируем уход, если клиент обращался в поддержку **4 раза или больше** **либо** подключил международный роуминг.
Положительный класс — уход клиента.

- Ошибка первого рода (FPR): FP / (FP + TN), доля ложных тревог среди фактически активных клиентов.
- Ошибка второго рода (FNR): FN / (FN + TP), доля пропущенных уходов среди фактически ушедших.

Дополнительно выведем доли FP и FN от всей выборки, чтобы отличать их от условных частот ошибок.
Правило выбрано и оценено на одних данных; это учебная оценка, а не проверка качества на новых клиентах.

In [8]:
df['Прогнозируемый отток'] = df['Customer service calls'].ge(4) | df['International plan'].eq('yes')
cm = pd.crosstab(df['Churn'], df['Прогнозируемый отток'], rownames=['Факт'], colnames=['Прогноз']).reindex(index=[False, True], columns=[False, True], fill_value=0)
display(cm)
tn, fp, fn, tp = cm.loc[False, False], cm.loc[False, True], cm.loc[True, False], cm.loc[True, True]
metrics = pd.Series({
    'Ошибка первого рода, FPR (%)': 100 * fp / (fp + tn),
    'Ошибка второго рода, FNR (%)': 100 * fn / (fn + tp),
    'FP от всех клиентов (%)': 100 * fp / len(df),
    'FN от всех клиентов (%)': 100 * fn / len(df),
    'Общая доля ошибок (%)': 100 * (fp + fn) / len(df),
    'Доля верных прогнозов (%)': 100 * (tp + tn) / len(df),
})
display(metrics.round(2).to_frame('Значение'))
print(f'Ложноположительных прогнозов: {fp}; ложноотрицательных: {fn}.')

Прогноз,False,True
Факт,,
False,2544,306
True,227,256


,Значение
"Ошибка первого рода, FPR (%)",10.74
"Ошибка второго рода, FNR (%)",47.00
FP от всех клиентов (%),9.18
FN от всех клиентов (%),6.81
Общая доля ошибок (%),15.99
Доля верных прогнозов (%),84.01


Ложноположительных прогнозов: 306; ложноотрицательных: 227.


**Вывод.** Правило даёт 10,74% ложных тревог среди активных клиентов и пропускает 47,00% фактических уходов. Общая точность 84,01% ниже точности постоянного прогноза «клиент остаётся» (85,51%), хотя постоянный прогноз пропускает все уходы. Это показывает, почему оценивать прогноз только по точности недостаточно.